# InfoStab Q1 Boundary-Class-Balanced No-GATE Benchmark

This Colab notebook removes the previous RBF-reliability losses and adds two new losses: `cb_focal_ca_msl_sg` and `boundary_cb_focal_ca_msl_sg`. It keeps validation-tuned hyperparameters, validation threshold tuning, and class-aware RBF center initialization.

In [ ]:

# =============================================================================
# InfoStab / Boundary-Class-Balanced CA-MSL-SG Q1 Validation-Tuned Benchmark, NO-GATE version
# 20 medical/complex real binary datasets + 5 synthetic stress-test datasets
#
# Changes vs previous infostab_q1_single_run_colab.ipynb:
#   1) All GATE-based loss methods are removed.
#   2) Loss hyperparameters are selected by validation-only tuning.
#   3) Sensitivity analysis is computed for key loss parameters.
#   4) Real and synthetic results are kept separate.
#   5) Test set is never used for tuning, threshold selection, or early stopping.
# =============================================================================

# -----------------------------
# 0. Imports
# -----------------------------
import os
import time
import json
import math
import hashlib
import random
import pickle
import zipfile
import warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from collections import OrderedDict
from typing import Dict, Any, List, Tuple, Optional

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt

from sklearn.datasets import (
    load_breast_cancer,
    make_moons,
    make_circles,
    make_classification,
    fetch_openml,
)
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from scipy.stats import wilcoxon


# =============================================================================
# 1. Configuration
# =============================================================================

@dataclass
class ExperimentConfig:
    output_name: str = "infostab_q1_boundary_classbalanced_no_gate"
    seed_base: int = 20260606
    use_gpu_if_available: bool = True

    # Main benchmark
    n_runs: int = 5
    final_epochs: int = 120
    final_patience: int = 30

    # Validation tuning
    tuning_mode: str = "validation"  # "validation" or "fixed"
    tuning_epochs: int = 80
    tuning_patience: int = 20
    tune_metric: str = "val_f1"      # val_f1, val_macro_f1, val_auc

    # Sensitivity analysis
    run_sensitivity_analysis: bool = True
    sensitivity_epochs: int = 70
    sensitivity_patience: int = 18
    sensitivity_run: int = 0

    # Practical batch option; default full run.
    run_dataset_batch_only: bool = False
    datasets_per_batch: int = 5
    dataset_batch_index: int = 0

    # Optimization
    lr: float = 5e-3
    weight_decay: float = 1e-5
    grad_clip: float = 1.0

    # Splits
    test_size: float = 0.15
    val_size_from_trainval: float = 0.12

    # RBF model
    max_centers: int = 80
    min_centers: int = 18
    center_fraction_divisor: int = 5
    sigma_min: float = 0.05

    # Preprocessing and runtime safety
    max_real_datasets: int = 20
    max_openml_rows: int = 5000
    max_features_after_preprocess: int = 160

    # Default loss parameters for fixed-global baseline and sensitivity centers.
    tau: float = 0.35
    beta: float = 4.5
    alpha: float = 0.30
    eps_abs: float = 1e-6
    responsibility_eps: float = 1e-8

    # Default InfoStab weights.
    lambda_entropy: float = 0.035
    lambda_balance: float = 0.035
    lambda_coverage: float = 0.0035
    lambda_radius: float = 1e-3

    # Class-balanced / hard-sample-aware CA-MSL-SG defaults.
    # These are only defaults; the new proposed losses are validation-tuned.
    class_balance_power: float = 0.50
    class_weight_clip: float = 3.0
    hard_gamma: float = 1.0
    minority_tau_multiplier: float = 1.10

    # Boundary-aware RBF up-weighting defaults. Unlike the previous reliability
    # weighting idea, this never down-weights low-coverage samples; it only
    # up-weights samples that are both covered by RBF kernels and geometrically ambiguous.
    boundary_lambda: float = 0.25
    boundary_rho_power: float = 1.0
    boundary_entropy_power: float = 1.0

    # RBF center initialization.
    # Class-aware centers reduce the risk that majority-class samples dominate the RBF basis.
    class_aware_centers: bool = True
    minority_center_min_fraction: float = 0.35

    # Threshold search on validation set
    validation_select_direction_and_threshold: bool = True
    threshold_quantiles: int = 81

    # Output
    save_history: bool = True
    history_stride: int = 10
    make_figures: bool = True
    auto_zip: bool = True


CFG = ExperimentConfig()

DEVICE = torch.device(
    "cuda" if CFG.use_gpu_if_available and torch.cuda.is_available() else "cpu"
)

TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path(f"/content/{CFG.output_name}_{TIMESTAMP}") if Path("/content").exists() else Path(f"./{CFG.output_name}_{TIMESTAMP}")
TABLE_DIR = OUTPUT_ROOT / "tables"
FIG_DIR = OUTPUT_ROOT / "figures"
LOG_DIR = OUTPUT_ROOT / "logs"
for d in [OUTPUT_ROOT, TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Output:", OUTPUT_ROOT)


# =============================================================================
# 2. Reproducibility utilities
# =============================================================================

def stable_seed(*parts: Any) -> int:
    s = "|".join(str(p) for p in parts)
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return (int(h, 16) + CFG.seed_base) % (2**31 - 1)


def set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


set_seeds(CFG.seed_base)


# =============================================================================
# 3. Dataset loading: 20 real binary + 5 synthetic
# =============================================================================

def _clean_frame_and_target(X, y) -> Tuple[pd.DataFrame, np.ndarray]:
    X_df = pd.DataFrame(X).copy()
    y_ser = pd.Series(y).copy()

    mask = ~y_ser.isna()
    X_df = X_df.loc[mask].reset_index(drop=True)
    y_ser = y_ser.loc[mask].reset_index(drop=True)

    X_df = X_df.dropna(axis=1, how="all")
    X_df = X_df.replace([np.inf, -np.inf], np.nan)

    vals = pd.Series(y_ser).dropna().astype(str).unique()
    if len(vals) != 2:
        raise ValueError(f"Target has {len(vals)} classes, expected 2.")

    le = LabelEncoder()
    y_enc = le.fit_transform(pd.Series(y_ser).astype(str)).astype(np.float32)

    counts = np.bincount(y_enc.astype(int))
    if len(counts) != 2 or np.min(counts) < 5:
        raise ValueError(f"Too few samples in one class after encoding: {counts}")

    return X_df, y_enc


def _subsample_if_needed(X_df: pd.DataFrame, y: np.ndarray, max_rows: int, seed: int):
    if len(y) <= max_rows:
        return X_df.reset_index(drop=True), y
    idx_all = np.arange(len(y))
    _, idx_sub = train_test_split(
        idx_all,
        test_size=max_rows,
        random_state=seed,
        stratify=y
    )
    idx_sub = np.sort(idx_sub)
    return X_df.iloc[idx_sub].reset_index(drop=True), y[idx_sub]


def load_sklearn_breast_cancer():
    ds = load_breast_cancer(as_frame=True)
    X_df = ds.frame.drop(columns=["target"])
    y = ds.frame["target"].values.astype(np.float32)
    return X_df, y


def load_openml_binary(dataset_key: str, names: List[str], max_rows: int = None):
    if max_rows is None:
        max_rows = CFG.max_openml_rows

    last_error = None
    for name in names:
        try:
            print(f"      trying OpenML name='{name}'")
            try:
                data = fetch_openml(name=name, version="active", as_frame=True, parser="auto")
            except TypeError:
                data = fetch_openml(name=name, version="active", as_frame=True)

            X_df, y = _clean_frame_and_target(data.data, data.target)
            X_df, y = _subsample_if_needed(X_df, y, max_rows, stable_seed(dataset_key, "subsample"))
            print(f"      success: {name} | n={len(y)} | d_raw={X_df.shape[1]}")
            return X_df, y
        except Exception as e:
            last_error = e
            print(f"      failed: {name} | {str(e)[:160]}")

    raise RuntimeError(f"All OpenML names failed for {dataset_key}. Last error: {last_error}")


REAL_REGISTRY = OrderedDict([
    # Medical / biomedical / diagnostic datasets first.
    # The loader is robust: if a name fails on OpenML, it tries the next alias and then moves on.
    ("breast_cancer_sklearn", ("sklearn", ["breast_cancer_sklearn"])),
    ("breast_w", ("openml", ["breast-w", "breast-cancer-wisconsin"])),
    # Note: OpenML WDBC is intentionally not included because it duplicates sklearn breast_cancer.
    ("pima_diabetes", ("openml", ["diabetes", "pima-indians-diabetes"])),
    ("heart_statlog", ("openml", ["heart-statlog", "heart-statlog-heart"])),
    ("blood_transfusion", ("openml", ["blood-transfusion-service-center", "blood-transfusion"])),
    ("ilpd", ("openml", ["ilpd", "Indian-Liver-Patient-Dataset"])),
    ("haberman", ("openml", ["haberman", "Haberman-Survival"])),
    ("hepatitis", ("openml", ["hepatitis"])),
    ("mammography", ("openml", ["mammography", "mammography_01"])),
    ("colic", ("openml", ["colic", "horse-colic"])),
    ("sick", ("openml", ["sick", "sick-euthyroid", "thyroid-sick"])),
    ("eeg_eye_state", ("openml", ["eeg-eye-state"])),
    ("qsar_biodeg", ("openml", ["qsar-biodeg"])),

    # Complex/noisy/high-dimensional real binary datasets.
    ("ionosphere", ("openml", ["ionosphere"])),
    ("sonar", ("openml", ["sonar"])),
    ("spambase", ("openml", ["spambase"])),
    ("phoneme", ("openml", ["phoneme"])),
    ("madelon", ("openml", ["madelon"])),
    ("hill_valley", ("openml", ["Hill_Valley_without_noise", "Hill_Valley_with_noise", "hill-valley"])),
    ("banknote_authentication", ("openml", ["banknote-authentication"])),
    ("credit_g", ("openml", ["credit-g"])),
    ("credit_approval", ("openml", ["credit-approval", "Australian"])),
    ("tic_tac_toe", ("openml", ["tic-tac-toe"])),
    ("kr_vs_kp", ("openml", ["kr-vs-kp"])),
    ("mushroom", ("openml", ["mushroom"])),
    ("climate_model_crashes", ("openml", ["climate-model-simulation-crashes", "climate-model-crashes"])),
    ("pc1", ("openml", ["pc1"])),
    ("kc1", ("openml", ["kc1"])),

    # Backup candidates; used only if one of the above fails before reaching 20.
    ("pc3", ("openml", ["pc3"])),
    ("pc4", ("openml", ["pc4"])),
    ("kc2", ("openml", ["kc2"])),
])


def load_20_real_datasets() -> Tuple[OrderedDict, pd.DataFrame]:
    tasks = OrderedDict()
    rows = []
    failures = []

    print("\n" + "=" * 80)
    print("LOADING MEDICAL / COMPLEX REAL BINARY DATASETS")
    print("=" * 80)

    for key, (src, names) in REAL_REGISTRY.items():
        if len(tasks) >= CFG.max_real_datasets:
            break
        print(f"\n[{len(tasks)+1}/{CFG.max_real_datasets}] {key}")
        try:
            if src == "sklearn":
                X_df, y = load_sklearn_breast_cancer()
            else:
                X_df, y = load_openml_binary(key, names)

            counts = np.bincount(y.astype(int))
            if len(counts) != 2 or np.min(counts) < 5:
                raise ValueError(f"Invalid class distribution: {counts}")

            tasks[key] = {
                "key": key,
                "source": "real",
                "X_raw": X_df.reset_index(drop=True),
                "y": y.astype(np.float32),
                "n": len(y),
                "raw_d": X_df.shape[1],
                "class0": int(counts[0]),
                "class1": int(counts[1]),
            }
            rows.append({
                "key": key,
                "source": "real",
                "n": len(y),
                "raw_d": X_df.shape[1],
                "class0": int(counts[0]),
                "class1": int(counts[1]),
                "status": "loaded",
            })
            print(f"      loaded real dataset: n={len(y)}, raw_d={X_df.shape[1]}, counts={counts.tolist()}")

        except Exception as e:
            failures.append({"key": key, "error": str(e)})
            print(f"      FAILED and skipped: {str(e)[:200]}")

    if len(tasks) < CFG.max_real_datasets:
        failure_df = pd.DataFrame(failures)
        failure_df.to_csv(TABLE_DIR / "real_dataset_failures.csv", index=False)
        raise RuntimeError(
            f"Only loaded {len(tasks)} real binary datasets; required {CFG.max_real_datasets}. "
            f"See {TABLE_DIR / 'real_dataset_failures.csv'}."
        )

    return tasks, pd.DataFrame(rows)


def load_5_synthetic_datasets() -> OrderedDict:
    print("\n" + "=" * 80)
    print("CREATING 5 SYNTHETIC STRESS-TEST DATASETS")
    print("=" * 80)

    syn = OrderedDict()

    X, y = make_moons(n_samples=1200, noise=0.28, random_state=stable_seed("syn_moons"))
    syn["syn_moons"] = {"key": "syn_moons", "source": "synthetic", "X_raw": pd.DataFrame(X), "y": y.astype(np.float32)}

    X, y = make_circles(n_samples=1200, noise=0.12, factor=0.45, random_state=stable_seed("syn_circles"))
    syn["syn_circles"] = {"key": "syn_circles", "source": "synthetic", "X_raw": pd.DataFrame(X), "y": y.astype(np.float32)}

    X, y = make_classification(
        n_samples=1500, n_features=20, n_informative=8, n_redundant=4,
        n_clusters_per_class=3, weights=[0.82, 0.18], class_sep=0.85,
        flip_y=0.04, random_state=stable_seed("syn_imbalanced")
    )
    syn["syn_imbalanced"] = {"key": "syn_imbalanced", "source": "synthetic", "X_raw": pd.DataFrame(X), "y": y.astype(np.float32)}

    X, y = make_classification(
        n_samples=1500, n_features=100, n_informative=12, n_redundant=8,
        n_clusters_per_class=2, class_sep=0.75, flip_y=0.05,
        random_state=stable_seed("syn_highdim")
    )
    syn["syn_highdim"] = {"key": "syn_highdim", "source": "synthetic", "X_raw": pd.DataFrame(X), "y": y.astype(np.float32)}

    X, y = make_classification(
        n_samples=1500, n_features=30, n_informative=10, n_redundant=5,
        n_clusters_per_class=2, class_sep=0.55, flip_y=0.12,
        random_state=stable_seed("syn_noisy_overlap")
    )
    syn["syn_noisy_overlap"] = {"key": "syn_noisy_overlap", "source": "synthetic", "X_raw": pd.DataFrame(X), "y": y.astype(np.float32)}

    for k, v in syn.items():
        yv = v["y"].astype(int)
        counts = np.bincount(yv)
        v["n"] = len(yv)
        v["raw_d"] = v["X_raw"].shape[1]
        v["class0"] = int(counts[0])
        v["class1"] = int(counts[1])
        print(f"      {k}: n={v['n']}, raw_d={v['raw_d']}, counts={counts.tolist()}")

    return syn


def load_all_datasets() -> Tuple[OrderedDict, pd.DataFrame]:
    real, _ = load_20_real_datasets()
    syn = load_5_synthetic_datasets()
    all_tasks = OrderedDict()
    all_tasks.update(real)
    all_tasks.update(syn)

    if CFG.run_dataset_batch_only:
        keys = list(all_tasks.keys())
        start = CFG.dataset_batch_index * CFG.datasets_per_batch
        end = start + CFG.datasets_per_batch
        selected_keys = keys[start:end]
        all_tasks = OrderedDict((k, all_tasks[k]) for k in selected_keys)
        print(f"\nDATASET BATCH MODE: batch={CFG.dataset_batch_index}, keys={selected_keys}")

    manifest_rows = []
    for k, task in all_tasks.items():
        manifest_rows.append({
            "key": k,
            "source": task["source"],
            "n": task["n"],
            "raw_d": task["raw_d"],
            "class0": task["class0"],
            "class1": task["class1"],
        })
    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(TABLE_DIR / "dataset_manifest.csv", index=False)
    return all_tasks, manifest


# =============================================================================
# 4. Preprocessing
# =============================================================================

@dataclass
class PreparedSplit:
    dataset_key: str
    source: str
    run: int
    X_train: np.ndarray
    X_val: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    feature_dim: int
    n_train: int
    n_val: int
    n_test: int
    raw_d: int
    class0: int
    class1: int


def _make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def _preprocess_fit_transform(X_train_df: pd.DataFrame, X_val_df: pd.DataFrame, X_test_df: pd.DataFrame):
    X_train_df = pd.DataFrame(X_train_df).reset_index(drop=True)
    X_val_df = pd.DataFrame(X_val_df).reset_index(drop=True)
    X_test_df = pd.DataFrame(X_test_df).reset_index(drop=True)

    cat_cols = []
    num_cols = []

    for col in X_train_df.columns:
        if str(X_train_df[col].dtype) in ["object", "category", "bool"]:
            cat_cols.append(col)
        else:
            tmp = pd.to_numeric(X_train_df[col], errors="coerce")
            if tmp.notna().mean() >= 0.80:
                num_cols.append(col)
                X_train_df[col] = pd.to_numeric(X_train_df[col], errors="coerce")
                X_val_df[col] = pd.to_numeric(X_val_df[col], errors="coerce")
                X_test_df[col] = pd.to_numeric(X_test_df[col], errors="coerce")
            else:
                cat_cols.append(col)

    transformers = []
    if num_cols:
        transformers.append((
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            num_cols
        ))
    if cat_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("onehot", _make_onehot_encoder()),
            ]),
            cat_cols
        ))

    if not transformers:
        raise ValueError("No usable columns after preprocessing.")

    pre = ColumnTransformer(transformers, remainder="drop")

    Xtr = pre.fit_transform(X_train_df)
    Xva = pre.transform(X_val_df)
    Xte = pre.transform(X_test_df)

    Xtr = np.asarray(Xtr, dtype=np.float32)
    Xva = np.asarray(Xva, dtype=np.float32)
    Xte = np.asarray(Xte, dtype=np.float32)

    if Xtr.shape[1] > CFG.max_features_after_preprocess:
        ncomp = min(CFG.max_features_after_preprocess, Xtr.shape[0] - 1, Xtr.shape[1])
        pca = PCA(n_components=ncomp, random_state=CFG.seed_base)
        Xtr = pca.fit_transform(Xtr).astype(np.float32)
        Xva = pca.transform(Xva).astype(np.float32)
        Xte = pca.transform(Xte).astype(np.float32)

    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xva = sc.transform(Xva).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)

    return Xtr, Xva, Xte


def prepare_split(task: Dict[str, Any], run: int) -> PreparedSplit:
    key = task["key"]
    X_df = pd.DataFrame(task["X_raw"]).reset_index(drop=True)
    y = np.asarray(task["y"]).astype(np.float32).ravel()

    seed = stable_seed(key, run, "split")
    idx = np.arange(len(y))
    idx_trainval, idx_test = train_test_split(
        idx,
        test_size=CFG.test_size,
        random_state=seed,
        stratify=y
    )
    y_trainval = y[idx_trainval]
    idx_train, idx_val = train_test_split(
        idx_trainval,
        test_size=CFG.val_size_from_trainval,
        random_state=seed,
        stratify=y_trainval
    )

    X_train_df = X_df.iloc[idx_train].reset_index(drop=True)
    X_val_df = X_df.iloc[idx_val].reset_index(drop=True)
    X_test_df = X_df.iloc[idx_test].reset_index(drop=True)

    y_train = y[idx_train].astype(np.float32)
    y_val = y[idx_val].astype(np.float32)
    y_test = y[idx_test].astype(np.float32)

    X_train, X_val, X_test = _preprocess_fit_transform(X_train_df, X_val_df, X_test_df)

    return PreparedSplit(
        dataset_key=key,
        source=task["source"],
        run=run,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        feature_dim=X_train.shape[1],
        n_train=len(y_train),
        n_val=len(y_val),
        n_test=len(y_test),
        raw_d=task["raw_d"],
        class0=task["class0"],
        class1=task["class1"],
    )


# =============================================================================
# 5. RBF network and initialization
# =============================================================================

def softplus_inverse(x: float) -> float:
    x = max(float(x), 1e-8)
    if x > 20:
        return x
    return math.log(math.exp(x) - 1.0)


def choose_num_centers(n_train: int) -> int:
    k = min(CFG.max_centers, max(CFG.min_centers, n_train // CFG.center_fraction_divisor))
    return int(min(k, n_train))


def robust_initial_sigma(X: np.ndarray, centers: np.ndarray) -> float:
    dist = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2))
    nearest = np.min(dist, axis=1)
    med = float(np.median(nearest))
    if not np.isfinite(med) or med <= 1e-8:
        med = 1.0
    return max(CFG.sigma_min + 1e-3, med)


class RBFNet(nn.Module):
    def __init__(self, input_dim: int, num_centers: int):
        super().__init__()
        self.input_dim = input_dim
        self.num_centers = num_centers

        self.centers = nn.Parameter(torch.zeros(num_centers, input_dim))
        self.weights = nn.Parameter(torch.zeros(num_centers))
        self.bias = nn.Parameter(torch.zeros(()))
        self.sigma_raw = nn.Parameter(torch.zeros(()))
        self.register_buffer("sigma0", torch.tensor(1.0, dtype=torch.float32))

        self.eval_direction = 1.0
        self.eval_threshold = 0.0
        self.register_buffer("sigma0_buffer", torch.tensor(1.0, dtype=torch.float32))

    def sigma(self):
        return CFG.sigma_min + F.softplus(self.sigma_raw)

    def forward(self, x: torch.Tensor, return_aux: bool = False):
        dist = torch.cdist(x, self.centers, p=2)
        dist2 = dist.pow(2)
        sig = self.sigma()
        phi = torch.exp(-dist2 / (2.0 * sig.pow(2) + 1e-8))
        score = phi @ self.weights + self.bias

        if return_aux:
            return score, {"dist2": dist2, "phi": phi, "sigma": sig, "sigma0": self.sigma0_buffer}
        return score


def _allocate_class_centers(y_train: np.ndarray, k: int) -> Tuple[int, int]:
    """
    Allocate RBF centers between the two classes with a minimum share for the minority class.
    This is important for medical/imbalanced datasets where global KMeans may under-cover
    the minority class.
    """
    y = np.asarray(y_train).astype(int)
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    if n0 <= 0 or n1 <= 0:
        return k, 0

    k = min(int(k), n0 + n1)
    min_frac = float(CFG.minority_center_min_fraction)

    # Proportional allocation, then enforce a minimum minority share.
    k0 = int(round(k * n0 / (n0 + n1)))
    k1 = k - k0

    if n0 <= n1:
        k0 = max(k0, int(round(k * min_frac)))
    else:
        k1 = max(k1, int(round(k * min_frac)))

    # Respect class sample counts and at least one center per non-empty class.
    k0 = min(max(1, k0), n0)
    k1 = min(max(1, k1), n1)

    # If we over-allocated, remove from the class with larger excess capacity.
    while k0 + k1 > k:
        if k0 > k1 and k0 > 1:
            k0 -= 1
        elif k1 > 1:
            k1 -= 1
        elif k0 > 1:
            k0 -= 1
        else:
            break

    # If we under-allocated, add to classes with remaining capacity.
    while k0 + k1 < k:
        add0 = n0 - k0
        add1 = n1 - k1
        if add0 <= 0 and add1 <= 0:
            break
        if add0 >= add1 and add0 > 0:
            k0 += 1
        elif add1 > 0:
            k1 += 1
        else:
            k0 += 1

    return int(k0), int(k1)


def _kmeans_or_all_points(X: np.ndarray, k: int, seed: int) -> np.ndarray:
    k = int(min(max(1, k), len(X)))
    if k >= len(X):
        return X.astype(np.float32).copy()
    km = KMeans(n_clusters=k, n_init=10, random_state=seed)
    return km.fit(X).cluster_centers_.astype(np.float32)


def create_initial_state(split: PreparedSplit) -> Dict[str, torch.Tensor]:
    seed = stable_seed(split.dataset_key, split.run, "init")
    set_seeds(seed)

    k = choose_num_centers(split.n_train)

    if CFG.class_aware_centers and len(np.unique(split.y_train.astype(int))) == 2:
        k0, k1 = _allocate_class_centers(split.y_train, k)
        X0 = split.X_train[split.y_train.astype(int) == 0]
        X1 = split.X_train[split.y_train.astype(int) == 1]
        c0 = _kmeans_or_all_points(X0, k0, stable_seed(split.dataset_key, split.run, "kmeans0"))
        c1 = _kmeans_or_all_points(X1, k1, stable_seed(split.dataset_key, split.run, "kmeans1"))
        centers = np.vstack([c0, c1]).astype(np.float32)

        # In rare cases k may be reduced by class sample limits. Fill remaining centers globally.
        if centers.shape[0] < k:
            extra = _kmeans_or_all_points(split.X_train, k - centers.shape[0], stable_seed(split.dataset_key, split.run, "kmeans_extra"))
            centers = np.vstack([centers, extra]).astype(np.float32)
        elif centers.shape[0] > k:
            centers = centers[:k]
    else:
        centers = _kmeans_or_all_points(split.X_train, k, seed)

    k = int(centers.shape[0])
    sigma0 = robust_initial_sigma(split.X_train, centers)

    rng = np.random.default_rng(seed)
    weights = rng.normal(0.0, math.sqrt(2.0 / max(1, k)), size=k).astype(np.float32)

    return {
        "centers": torch.tensor(centers, dtype=torch.float32),
        "weights": torch.tensor(weights, dtype=torch.float32),
        "bias": torch.tensor(0.0, dtype=torch.float32),
        "sigma_raw": torch.tensor(softplus_inverse(sigma0 - CFG.sigma_min), dtype=torch.float32),
        "num_centers": torch.tensor(k),
        "sigma0": torch.tensor(sigma0),
    }


def instantiate_model(split: PreparedSplit, init_state: Dict[str, torch.Tensor]) -> RBFNet:
    k = int(init_state["num_centers"].item())
    model = RBFNet(split.feature_dim, k)
    with torch.no_grad():
        model.centers.copy_(init_state["centers"])
        model.weights.copy_(init_state["weights"])
        model.bias.copy_(init_state["bias"])
        model.sigma_raw.copy_(init_state["sigma_raw"])
        model.sigma0_buffer.copy_(init_state["sigma0"].float())
    return model.to(DEVICE)


# =============================================================================
# 6. Losses and InfoStab objective
# =============================================================================

@dataclass
class MethodConfig:
    key: str
    display_name: str
    base_loss: str
    use_infostab: bool = False
    use_entropy: bool = False
    use_balance: bool = False
    use_coverage: bool = False
    stop_gradient_threshold: bool = False

    tau: float = CFG.tau
    beta: float = CFG.beta
    alpha: float = CFG.alpha

    lambda_entropy: float = CFG.lambda_entropy
    lambda_balance: float = CFG.lambda_balance
    lambda_coverage: float = CFG.lambda_coverage
    lambda_radius: float = CFG.lambda_radius

    # New class-balanced / hard-sample-aware parameters.
    use_class_balance: bool = False
    use_hard_focus: bool = False
    use_class_tau: bool = False
    use_boundary_weight: bool = False
    class_balance_power: float = CFG.class_balance_power
    class_weight_clip: float = CFG.class_weight_clip
    hard_gamma: float = CFG.hard_gamma
    minority_tau_multiplier: float = CFG.minority_tau_multiplier
    boundary_lambda: float = CFG.boundary_lambda
    boundary_rho_power: float = CFG.boundary_rho_power
    boundary_entropy_power: float = CFG.boundary_entropy_power

    focal_gamma: float = 2.0
    huber_delta: float = 1.0
    candidate_id: str = "default"


def abs_eps(x: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(x.pow(2) + CFG.eps_abs * CFG.eps_abs)


def softplus_beta_from_z(z: torch.Tensor, beta: float) -> torch.Tensor:
    return F.softplus(z) / beta



def rbf_boundary_terms(phi: torch.Tensor, method: MethodConfig):
    """
    Boundary signal for RBF networks.

    rho_i: max_k phi_k(x_i), high when the sample is covered by at least one RBF unit.
    H_i: normalized responsibility entropy, high when several centers compete.
    boundary_i = rho_i^a * H_i^b, high for covered-but-ambiguous boundary samples.

    Important: unlike the previous reliability weighting, this term never down-weights
    low-coverage samples. It can only up-weight likely boundary samples.
    """
    eps = CFG.responsibility_eps
    phi_safe = torch.clamp(phi, min=eps)
    P = phi_safe / torch.clamp(phi_safe.sum(dim=1, keepdim=True), min=eps)
    K = max(int(phi.shape[1]), 2)

    rho = torch.clamp(phi_safe.max(dim=1).values, min=eps, max=1.0)
    H = -(P * torch.log(P + eps)).sum(dim=1) / math.log(K)
    H = torch.clamp(H, min=0.0, max=1.0)

    boundary = (rho ** method.boundary_rho_power) * (H ** method.boundary_entropy_power)
    boundary = torch.clamp(boundary, min=0.0, max=1.0)
    return P, rho, H, boundary


def class_balanced_hard_ca_msl_sg_loss(
    scores: torch.Tensor,
    y01: torch.Tensor,
    aux: Dict[str, torch.Tensor],
    method: MethodConfig,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    New loss family replacing the previous RBF-reliability losses.

    1) cb_focal_ca_msl_sg:
       CA-MSL-SG + class-balanced weighting + hard-sample focusing.

    2) boundary_cb_focal_ca_msl_sg:
       cb_focal_ca_msl_sg + RBF boundary up-weighting.

    Core idea:
      - Do not down-weight low-coverage samples, because in medical/imbalanced data
        they may be important minority-class samples.
      - Instead, compensate class imbalance and focus on hard / low-margin samples.
      - Optionally up-weight samples that are both RBF-covered and center-ambiguous.
    """
    yf = y01.float()
    ys = 2.0 * yf - 1.0
    eps = CFG.responsibility_eps

    n0 = torch.clamp((yf < 0.5).float().sum(), min=1.0)
    n1 = torch.clamp((yf >= 0.5).float().sum(), min=1.0)
    n = n0 + n1

    # Class-balanced sample weights. power=0 disables the effect; power=0.5 is mild.
    w0 = (n / (2.0 * n0)).pow(method.class_balance_power)
    w1 = (n / (2.0 * n1)).pow(method.class_balance_power)
    sample_w = torch.where(yf >= 0.5, w1, w0)
    sample_w = torch.clamp(sample_w, min=1.0 / method.class_weight_clip, max=method.class_weight_clip)
    sample_w = sample_w / torch.clamp(sample_w.mean().detach(), min=eps)

    # Focal-style hard-sample focusing in signed-margin probability space.
    pt = torch.sigmoid(ys * scores)
    hard_w = (1.0 - pt).clamp(min=0.0, max=1.0).pow(method.hard_gamma)
    if method.hard_gamma <= 0:
        hard_w = torch.ones_like(hard_w)

    omega = sample_w * hard_w

    # Boundary-aware RBF up-weighting: only increases weights for covered ambiguous samples.
    boundary = torch.zeros_like(scores)
    rho = torch.zeros_like(scores)
    H = torch.zeros_like(scores)
    boundary_w = torch.ones_like(scores)
    if method.use_boundary_weight and "phi" in aux:
        _, rho, H, boundary = rbf_boundary_terms(aux["phi"], method)
        boundary_w = 1.0 + method.boundary_lambda * boundary.detach()
        omega = omega * boundary_w

    # CA-MSL-SG adaptive margin. The threshold path uses stop-gradient.
    s_sg = scores.detach()
    tau_eff = method.tau * torch.exp(-method.alpha * abs_eps(s_sg))

    # Optional class-specific margin: ask a slightly larger margin from the minority class.
    if method.use_class_tau and method.minority_tau_multiplier != 1.0:
        minority_is_one = (n1 < n0)
        minority_mask = (yf >= 0.5) if bool(minority_is_one.item()) else (yf < 0.5)
        tau_eff = tau_eff * torch.where(
            minority_mask,
            torch.full_like(tau_eff, method.minority_tau_multiplier),
            torch.ones_like(tau_eff),
        )

    margin = ys * scores
    z = method.beta * (tau_eff - margin)
    loss_i = softplus_beta_from_z(z, method.beta)

    omega = torch.clamp(omega, min=eps)
    loss = (omega * loss_i).sum() / torch.clamp(omega.sum(), min=eps)

    stats = {
        "sample_weight_mean": float(sample_w.detach().mean().cpu()),
        "hard_weight_mean": float(hard_w.detach().mean().cpu()),
        "boundary_weight_mean": float(boundary_w.detach().mean().cpu()),
        "boundary_score_mean": float(boundary.detach().mean().cpu()),
        "rbf_rho_mean": float(rho.detach().mean().cpu()),
        "rbf_entropy_mean": float(H.detach().mean().cpu()),
        "tau_eff_mean": float(tau_eff.detach().mean().cpu()),
    }
    return loss, stats


def scalar_loss(scores: torch.Tensor, y01: torch.Tensor, method: MethodConfig) -> torch.Tensor:
    yf = y01.float()
    ys = 2.0 * yf - 1.0

    if method.base_loss == "bce":
        return F.binary_cross_entropy_with_logits(scores, yf)

    if method.base_loss == "mse_signed":
        return torch.mean((scores - ys).pow(2))

    if method.base_loss == "huber_signed":
        return F.smooth_l1_loss(scores, ys, beta=method.huber_delta)

    if method.base_loss == "focal":
        bce = F.binary_cross_entropy_with_logits(scores, yf, reduction="none")
        p = torch.sigmoid(scores)
        pt = torch.where(y01 == 1, p, 1.0 - p)
        return torch.mean((1.0 - pt).pow(method.focal_gamma) * bce)

    if method.base_loss == "cb_focal_ca_msl_sg":
        loss, _ = class_balanced_hard_ca_msl_sg_loss(scores, y01, {}, method)
        return loss

    if method.base_loss == "dice":
        p = torch.sigmoid(scores)
        eps = 1e-6
        inter = torch.sum(p * yf)
        denom = torch.sum(p) + torch.sum(yf)
        return 1.0 - (2.0 * inter + eps) / (denom + eps)

    if method.base_loss == "residual_tsl":
        r = scores - ys
        z = method.beta * (abs_eps(r) - method.tau)
        return torch.mean(softplus_beta_from_z(z, method.beta))

    if method.base_loss == "at_tsl":
        r = scores - ys
        tau_i = method.tau * torch.exp(-method.alpha * abs_eps(scores))
        z = method.beta * (abs_eps(r) - tau_i)
        return torch.mean(softplus_beta_from_z(z, method.beta))

    if method.base_loss == "msl":
        margin = ys * scores
        z = method.beta * (method.tau - margin)
        return torch.mean(softplus_beta_from_z(z, method.beta))

    if method.base_loss == "ca_msl":
        conf = scores.detach() if method.stop_gradient_threshold else scores
        tau_i = method.tau * torch.exp(-method.alpha * abs_eps(conf))
        margin = ys * scores
        z = method.beta * (tau_i - margin)
        return torch.mean(softplus_beta_from_z(z, method.beta))

    raise ValueError(f"Unknown base_loss: {method.base_loss}")


def responsibilities(phi: torch.Tensor) -> torch.Tensor:
    raw = phi + CFG.responsibility_eps
    return raw / torch.clamp(raw.sum(dim=1, keepdim=True), min=CFG.responsibility_eps)


def conditional_entropy(P: torch.Tensor, y01: torch.Tensor) -> torch.Tensor:
    total = torch.tensor(0.0, device=P.device)
    for cls in [0, 1]:
        mask = (y01 == cls)
        if mask.sum() == 0:
            continue
        pbar = P[mask].mean(dim=0)
        h = -(pbar * torch.log(pbar + CFG.responsibility_eps)).sum()
        total = total + mask.float().mean() * h
    return total


def smoothed_kl_balance(P: torch.Tensor) -> torch.Tensor:
    pbar = P.mean(dim=0)
    k = P.shape[1]
    ps = (pbar + CFG.responsibility_eps) / torch.clamp(
        pbar.sum() + k * CFG.responsibility_eps,
        min=CFG.responsibility_eps
    )
    u = torch.full_like(ps, 1.0 / k)
    return torch.sum(ps * torch.log(torch.clamp(ps / u, min=CFG.responsibility_eps)))


def soft_coverage(dist2: torch.Tensor, method: MethodConfig) -> torch.Tensor:
    with torch.no_grad():
        med = torch.median(dist2.detach())
        nu = torch.clamp(0.5 * med, min=1e-3)
    A = torch.softmax(-dist2.t() / nu, dim=1)
    radii = (A * dist2.t()).sum(dim=1)
    return torch.mean((radii - radii.mean()).pow(2)) + method.lambda_radius * radii.mean()


def total_objective(model: RBFNet, xb: torch.Tensor, yb: torch.Tensor, method: MethodConfig):
    scores, aux = model(xb, return_aux=True)

    rbf_stats = {}
    if method.base_loss in ["cb_focal_ca_msl_sg", "boundary_cb_focal_ca_msl_sg"]:
        loss_cls, rbf_stats = class_balanced_hard_ca_msl_sg_loss(scores, yb, aux, method)
    else:
        loss_cls = scalar_loss(scores, yb, method)
    total = loss_cls

    reg_entropy = torch.tensor(0.0, device=xb.device)
    reg_balance = torch.tensor(0.0, device=xb.device)
    reg_coverage = torch.tensor(0.0, device=xb.device)

    if method.use_infostab:
        P = responsibilities(aux["phi"])

        if method.use_entropy:
            reg_entropy = conditional_entropy(P, yb)
            total = total - method.lambda_entropy * reg_entropy

        if method.use_balance:
            reg_balance = smoothed_kl_balance(P)
            total = total + method.lambda_balance * reg_balance

        if method.use_coverage:
            reg_coverage = soft_coverage(aux["dist2"], method)
            total = total + method.lambda_coverage * reg_coverage

    stats = {
        "loss_cls": float(loss_cls.detach().cpu()),
        "reg_entropy": float(reg_entropy.detach().cpu()),
        "reg_balance": float(reg_balance.detach().cpu()),
        "reg_coverage": float(reg_coverage.detach().cpu()),
        "loss_total": float(total.detach().cpu()),
        "sigma": float(aux["sigma"].detach().cpu()),
        "rbf_rho_mean": rbf_stats.get("rbf_rho_mean", 0.0),
        "rbf_entropy_mean": rbf_stats.get("rbf_entropy_mean", 0.0),
        "sample_weight_mean": rbf_stats.get("sample_weight_mean", 0.0),
        "hard_weight_mean": rbf_stats.get("hard_weight_mean", 0.0),
        "boundary_weight_mean": rbf_stats.get("boundary_weight_mean", 0.0),
        "boundary_score_mean": rbf_stats.get("boundary_score_mean", 0.0),
        "tau_eff_mean": rbf_stats.get("tau_eff_mean", 0.0),
    }

    return total, stats


BASE_METHODS = [
    MethodConfig("bce", "BCE", "bce"),
    MethodConfig("mse_signed", "MSE signed", "mse_signed"),
    MethodConfig("huber_signed", "Huber signed", "huber_signed"),
    MethodConfig("focal", "Focal", "focal"),
    MethodConfig("dice", "Dice", "dice"),

    MethodConfig("residual_tsl", "Residual TSL", "residual_tsl"),
    MethodConfig("at_tsl", "AT-TSL", "at_tsl"),
    MethodConfig("infostab_tsl", "InfoStab-TSL", "residual_tsl", use_infostab=True, use_entropy=True, use_balance=True, use_coverage=True),
    MethodConfig("infostab_at_tsl", "InfoStab-AT-TSL", "at_tsl", use_infostab=True, use_entropy=True, use_balance=True, use_coverage=True),

    MethodConfig("msl", "MSL", "msl"),
    MethodConfig("ca_msl", "CA-MSL", "ca_msl"),
    MethodConfig("ca_msl_sg", "CA-MSL stop-gradient", "ca_msl", stop_gradient_threshold=True),
    MethodConfig(
        "cb_focal_ca_msl_sg",
        "CB-Focal CA-MSL-SG",
        "cb_focal_ca_msl_sg",
        stop_gradient_threshold=True,
        use_class_balance=True,
        use_hard_focus=True,
        use_class_tau=True,
    ),
    MethodConfig(
        "boundary_cb_focal_ca_msl_sg",
        "Boundary-CB-Focal CA-MSL-SG",
        "boundary_cb_focal_ca_msl_sg",
        stop_gradient_threshold=True,
        use_class_balance=True,
        use_hard_focus=True,
        use_class_tau=True,
        use_boundary_weight=True,
    ),
    MethodConfig("infostab_ca_msl", "InfoStab-CA-MSL", "ca_msl", use_infostab=True, use_entropy=True, use_balance=True, use_coverage=True),
]


# =============================================================================
# 7. Validation-tuned hyperparameter candidates
# =============================================================================

def _cid(prefix: str, **params) -> str:
    parts = [prefix]
    for k in sorted(params.keys()):
        v = params[k]
        if isinstance(v, float):
            parts.append(f"{k}{v:g}")
        else:
            parts.append(f"{k}{v}")
    return "_".join(parts).replace(".", "p").replace("-", "m")


def clone_method(m: MethodConfig, candidate_id: str, **updates) -> MethodConfig:
    return replace(m, candidate_id=candidate_id, **updates)


def with_lambda_scale(m: MethodConfig, scale: float, **updates) -> MethodConfig:
    return clone_method(
        m,
        candidate_id=_cid("ls", scale=scale, **updates),
        lambda_entropy=CFG.lambda_entropy * scale,
        lambda_balance=CFG.lambda_balance * scale,
        lambda_coverage=CFG.lambda_coverage * scale,
        **updates
    )


def fixed_candidate(m: MethodConfig) -> List[MethodConfig]:
    return [clone_method(m, "fixed")]


def get_validation_candidates(m: MethodConfig) -> List[MethodConfig]:
    if CFG.tuning_mode == "fixed":
        return fixed_candidate(m)

    key = m.key

    if key in ["bce", "mse_signed", "dice"]:
        return fixed_candidate(m)

    if key == "huber_signed":
        return [
            clone_method(m, _cid("huber", delta=d), huber_delta=d)
            for d in [0.5, 1.0]
        ]

    if key == "focal":
        return [
            clone_method(m, _cid("focal", gamma=g), focal_gamma=g)
            for g in [1.0, 2.0]
        ]

    if key in ["residual_tsl", "msl"]:
        presets = [
            {"tau": 0.25, "beta": 3.0},
            {"tau": 0.35, "beta": 4.5},
            {"tau": 0.50, "beta": 4.5},
        ]
        return [clone_method(m, _cid(key, **p), **p) for p in presets]

    if key in ["at_tsl", "ca_msl", "ca_msl_sg"]:
        presets = [
            {"tau": 0.25, "beta": 3.0, "alpha": 0.15},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.15},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.30},
            {"tau": 0.50, "beta": 4.5, "alpha": 0.40},
        ]
        return [clone_method(m, _cid(key, **p), **p) for p in presets]


    if key == "cb_focal_ca_msl_sg":
        presets = [
            {"tau": 0.25, "beta": 3.0, "alpha": 0.15, "class_balance_power": 0.50, "hard_gamma": 0.50, "minority_tau_multiplier": 1.00},
            {"tau": 0.25, "beta": 3.0, "alpha": 0.15, "class_balance_power": 0.50, "hard_gamma": 1.00, "minority_tau_multiplier": 1.05},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.30, "class_balance_power": 0.50, "hard_gamma": 1.00, "minority_tau_multiplier": 1.10},
            {"tau": 0.35, "beta": 2.5, "alpha": 0.30, "class_balance_power": 0.50, "hard_gamma": 1.50, "minority_tau_multiplier": 1.10},
            {"tau": 0.50, "beta": 4.5, "alpha": 0.40, "class_balance_power": 0.50, "hard_gamma": 1.00, "minority_tau_multiplier": 1.15},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.30, "class_balance_power": 0.25, "hard_gamma": 1.00, "minority_tau_multiplier": 1.05},
        ]
        return [clone_method(m, _cid(key, **p), **p) for p in presets]

    if key == "boundary_cb_focal_ca_msl_sg":
        presets = [
            {"tau": 0.25, "beta": 3.0, "alpha": 0.15, "class_balance_power": 0.50, "hard_gamma": 0.50, "minority_tau_multiplier": 1.00, "boundary_lambda": 0.10, "boundary_rho_power": 1.0, "boundary_entropy_power": 1.0},
            {"tau": 0.25, "beta": 3.0, "alpha": 0.15, "class_balance_power": 0.50, "hard_gamma": 1.00, "minority_tau_multiplier": 1.05, "boundary_lambda": 0.15, "boundary_rho_power": 1.0, "boundary_entropy_power": 1.0},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.30, "class_balance_power": 0.50, "hard_gamma": 1.00, "minority_tau_multiplier": 1.10, "boundary_lambda": 0.25, "boundary_rho_power": 1.0, "boundary_entropy_power": 1.0},
            {"tau": 0.35, "beta": 2.5, "alpha": 0.30, "class_balance_power": 0.50, "hard_gamma": 1.50, "minority_tau_multiplier": 1.10, "boundary_lambda": 0.25, "boundary_rho_power": 1.0, "boundary_entropy_power": 1.0},
            {"tau": 0.50, "beta": 4.5, "alpha": 0.40, "class_balance_power": 0.50, "hard_gamma": 1.00, "minority_tau_multiplier": 1.15, "boundary_lambda": 0.35, "boundary_rho_power": 1.0, "boundary_entropy_power": 1.0},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.30, "class_balance_power": 0.25, "hard_gamma": 1.00, "minority_tau_multiplier": 1.05, "boundary_lambda": 0.15, "boundary_rho_power": 1.0, "boundary_entropy_power": 0.5},
        ]
        return [clone_method(m, _cid(key, **p), **p) for p in presets]

    if key == "infostab_tsl":
        presets = [
            {"tau": 0.25, "beta": 3.0, "scale": 0.10},
            {"tau": 0.35, "beta": 4.5, "scale": 0.25},
            {"tau": 0.35, "beta": 4.5, "scale": 0.50},
            {"tau": 0.50, "beta": 4.5, "scale": 1.00},
        ]
        out = []
        for p in presets:
            scale = p.pop("scale")
            out.append(with_lambda_scale(m, scale, **p))
        return out

    if key in ["infostab_at_tsl", "infostab_ca_msl"]:
        presets = [
            {"tau": 0.25, "beta": 3.0, "alpha": 0.15, "scale": 0.10},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.15, "scale": 0.25},
            {"tau": 0.35, "beta": 4.5, "alpha": 0.30, "scale": 0.50},
            {"tau": 0.50, "beta": 4.5, "alpha": 0.40, "scale": 1.00},
        ]
        out = []
        for p in presets:
            scale = p.pop("scale")
            out.append(with_lambda_scale(m, scale, **p))
        return out

    return fixed_candidate(m)


def method_params_json(m: MethodConfig) -> str:
    d = asdict(m)
    keep = [
        "tau", "beta", "alpha",
        "lambda_entropy", "lambda_balance", "lambda_coverage", "lambda_radius",
        "use_class_balance", "use_hard_focus", "use_class_tau", "use_boundary_weight",
        "class_balance_power", "class_weight_clip", "hard_gamma", "minority_tau_multiplier",
        "boundary_lambda", "boundary_rho_power", "boundary_entropy_power",
        "focal_gamma", "huber_delta", "candidate_id"
    ]
    return json.dumps({k: d[k] for k in keep if k in d}, sort_keys=True)


def candidate_summary_table() -> pd.DataFrame:
    rows = []
    for m in BASE_METHODS:
        cands = get_validation_candidates(m)
        for c in cands:
            rows.append({
                "method": m.key,
                "display_name": m.display_name,
                "candidate_id": c.candidate_id,
                "params": method_params_json(c),
            })
    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "candidate_grid.csv", index=False)
    return df


# =============================================================================
# 8. Training and evaluation
# =============================================================================

def validation_direction_threshold(scores: np.ndarray, y_true: np.ndarray) -> Tuple[float, float]:
    scores = np.asarray(scores).ravel().astype(float)
    y_true = np.asarray(y_true).ravel().astype(int)

    if not CFG.validation_select_direction_and_threshold or len(np.unique(y_true)) < 2:
        return 1.0, 0.0

    best = (-1.0, 1.0, 0.0)
    for direction in [1.0, -1.0]:
        s = direction * scores
        qs = np.linspace(0.02, 0.98, CFG.threshold_quantiles)
        thresholds = np.unique(np.quantile(s, qs))
        thresholds = np.concatenate(([s.min() - 1e-6, 0.0], thresholds, [s.max() + 1e-6]))
        for th in thresholds:
            preds = (s >= th).astype(int)
            val = f1_score(y_true, preds, zero_division=0)
            if val > best[0]:
                best = (val, direction, float(th))
    return best[1], best[2]


def evaluate_scores(scores_raw: np.ndarray, y_true: np.ndarray, direction: float, threshold: float) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int).ravel()
    scores = direction * np.asarray(scores_raw).astype(float).ravel()
    probs = 1.0 / (1.0 + np.exp(-scores))
    preds = (scores >= threshold).astype(int)

    if len(np.unique(y_true)) < 2:
        auc = 0.5
    else:
        try:
            auc = roc_auc_score(y_true, probs)
        except Exception:
            auc = 0.5

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    except Exception:
        tn = fp = fn = tp = 0
        specificity = 0.0

    return {
        "accuracy": accuracy_score(y_true, preds),
        "balanced_accuracy": balanced_accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, preds, zero_division=0),
        "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, preds),
        "auc": auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def train_candidate(
    split: PreparedSplit,
    init_state: Dict[str, torch.Tensor],
    method: MethodConfig,
    epochs: int,
    patience: int,
    stage: str = "tune",
    collect_history: bool = True
):
    set_seeds(stable_seed(split.dataset_key, split.run, method.key, method.candidate_id, stage))

    model = instantiate_model(split, init_state)
    opt = optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10, min_lr=1e-6)

    Xtr = torch.tensor(split.X_train, dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(split.y_train, dtype=torch.float32, device=DEVICE)
    Xva = torch.tensor(split.X_val, dtype=torch.float32, device=DEVICE)
    yva = torch.tensor(split.y_val, dtype=torch.float32, device=DEVICE)

    best_val = float("inf")
    best_state = None
    no_improve = 0
    history_rows = []

    last_epoch = 0
    for epoch in range(epochs):
        last_epoch = epoch
        model.train()
        opt.zero_grad(set_to_none=True)
        loss, train_stats = total_objective(model, Xtr, ytr, method)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss, val_stats = total_objective(model, Xva, yva, method)
            val_value = float(val_loss.detach().cpu())
        scheduler.step(val_value)

        if CFG.save_history and collect_history and (epoch % CFG.history_stride == 0 or epoch == epochs - 1):
            history_rows.append({
                "dataset": split.dataset_key,
                "source": split.source,
                "run": split.run,
                "method": method.key,
                "candidate_id": method.candidate_id,
                "stage": stage,
                "epoch": epoch,
                "train_loss": float(loss.detach().cpu()),
                "val_loss": val_value,
                **{f"train_{k}": v for k, v in train_stats.items()},
                **{f"val_{k}": v for k, v in val_stats.items()},
            })

        if val_value < best_val - 1e-8:
            best_val = val_value
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        val_scores = model(torch.tensor(split.X_val, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy()
        test_scores = model(torch.tensor(split.X_test, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy()
        train_scores = model(torch.tensor(split.X_train, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy()
        _, aux_train = model(torch.tensor(split.X_train, dtype=torch.float32, device=DEVICE), return_aux=True)

    direction, threshold = validation_direction_threshold(val_scores, split.y_val)
    val_metrics = evaluate_scores(val_scores, split.y_val, direction, threshold)
    test_metrics = evaluate_scores(test_scores, split.y_test, direction, threshold)
    train_metrics = evaluate_scores(train_scores, split.y_train, direction, threshold)

    phi = aux_train["phi"].detach().cpu().numpy()
    active_centers = int((phi.mean(axis=0) > 0.01).sum())
    dead_centers = int((phi.mean(axis=0) <= 0.01).sum())
    final_sigma = float(aux_train["sigma"].detach().cpu())

    result = {
        "dataset": split.dataset_key,
        "source": split.source,
        "run": split.run,
        "method": method.key,
        "display_name": method.display_name,
        "candidate_id": method.candidate_id,
        "selected_params": method_params_json(method),
        "base_loss": method.base_loss,
        "use_infostab": method.use_infostab,
        "n_train": split.n_train,
        "n_val": split.n_val,
        "n_test": split.n_test,
        "raw_d": split.raw_d,
        "feature_dim": split.feature_dim,
        "num_centers": int(init_state["num_centers"].item()),
        "direction": direction,
        "threshold": threshold,
        "best_val_loss": best_val,
        "epochs_ran": last_epoch + 1,
        "final_sigma": final_sigma,
        "active_centers": active_centers,
        "dead_centers": dead_centers,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }

    return result, pd.DataFrame(history_rows)


def select_best_candidate(candidate_results: List[Dict[str, Any]]) -> int:
    metric = CFG.tune_metric
    vals = np.array([r.get(metric, -np.inf) for r in candidate_results], dtype=float)
    # Tie-breakers: validation AUC, validation loss, then smaller candidate index.
    best_val = np.nanmax(vals)
    candidates = np.where(vals == best_val)[0]
    if len(candidates) == 1:
        return int(candidates[0])

    best_idx = int(candidates[0])
    best_tuple = None
    for idx in candidates:
        r = candidate_results[int(idx)]
        tup = (r.get("val_auc", -np.inf), -r.get("best_val_loss", np.inf), -idx)
        if best_tuple is None or tup > best_tuple:
            best_tuple = tup
            best_idx = int(idx)
    return best_idx


# =============================================================================
# 9. Experiment runner with validation tuning
# =============================================================================

def run_full_experiment(tasks: OrderedDict) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    selected_results = []
    all_candidate_results = []
    all_history = []

    total_base = len(tasks) * CFG.n_runs * len(BASE_METHODS)
    done_base = 0
    t_start = time.time()

    candidate_grid = candidate_summary_table()
    print("\nCandidate grid by method:")
    print(candidate_grid.groupby("method")["candidate_id"].count())

    for dataset_key, task in tasks.items():
        for run in range(CFG.n_runs):
            print("\n" + "=" * 100)
            print(f"Dataset={dataset_key} | source={task['source']} | run={run+1}/{CFG.n_runs}")
            print("=" * 100)

            split = prepare_split(task, run)
            init_state = create_initial_state(split)

            print(
                f"Prepared: n_train={split.n_train}, n_val={split.n_val}, n_test={split.n_test}, "
                f"d={split.feature_dim}, centers={int(init_state['num_centers'])}, "
                f"sigma0={float(init_state['sigma0']):.4f}"
            )

            for base_method in BASE_METHODS:
                done_base += 1
                candidates = get_validation_candidates(base_method)
                print(f"\n[{done_base:4d}/{total_base}] Tuning method={base_method.key}, candidates={len(candidates)}")

                cand_results = []
                cand_histories = []

                for ci, cand in enumerate(candidates, 1):
                    t0 = time.time()
                    try:
                        res, hist = train_candidate(
                            split,
                            init_state,
                            cand,
                            epochs=CFG.tuning_epochs,
                            patience=CFG.tuning_patience,
                            stage="tune",
                            collect_history=True
                        )
                        elapsed = time.time() - t0
                        res["candidate_index"] = ci
                        res["num_candidates"] = len(candidates)
                        res["time_sec"] = elapsed
                        cand_results.append(res)
                        if not hist.empty:
                            cand_histories.append(hist)
                        print(
                            f"    cand {ci:02d}/{len(candidates):02d} {cand.candidate_id:<34} "
                            f"valF1={res['val_f1']:.4f} testF1={res['test_f1']:.4f} "
                            f"AUC={res['test_auc']:.4f} σ={res['final_sigma']:.3f} time={elapsed:.1f}s"
                        )
                    except Exception as e:
                        print(f"    [ERROR] candidate {cand.candidate_id}: {e}")

                if not cand_results:
                    print(f"    [SKIP] No successful candidates for {base_method.key}")
                    continue

                best_idx = select_best_candidate(cand_results)
                best_res = dict(cand_results[best_idx])
                best_res["is_selected"] = True
                best_res["tuning_metric"] = CFG.tune_metric
                best_res["selected_candidate_rank"] = best_idx + 1

                # Candidate audit table: include all candidates, selected flag.
                for i, cr in enumerate(cand_results):
                    cr = dict(cr)
                    cr["is_selected"] = (i == best_idx)
                    cr["tuning_metric"] = CFG.tune_metric
                    all_candidate_results.append(cr)

                selected_results.append(best_res)

                if cand_histories:
                    # Keep all tuning histories for diagnostics and sensitivity to hyperparameter behavior.
                    all_history.append(pd.concat(cand_histories, ignore_index=True))

                print(
                    f"    SELECTED {best_res['candidate_id']} | "
                    f"valF1={best_res['val_f1']:.4f} testF1={best_res['test_f1']:.4f} "
                    f"macroF1={best_res['test_macro_f1']:.4f} AUC={best_res['test_auc']:.4f}"
                )

                # Partial checkpoint after every base method.
                pd.DataFrame(selected_results).to_csv(TABLE_DIR / "raw_results_partial.csv", index=False)
                pd.DataFrame(all_candidate_results).to_csv(TABLE_DIR / "candidate_results_partial.csv", index=False)
                if all_history:
                    pd.concat(all_history, ignore_index=True).to_csv(TABLE_DIR / "history_partial.csv", index=False)

    raw = pd.DataFrame(selected_results)
    candidate_df = pd.DataFrame(all_candidate_results)
    history = pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()

    raw.to_csv(TABLE_DIR / "raw_results.csv", index=False)
    candidate_df.to_csv(TABLE_DIR / "candidate_results.csv", index=False)
    history.to_csv(TABLE_DIR / "history.csv", index=False)

    print("\nTotal experiment time:", time.time() - t_start, "seconds")
    return raw, candidate_df, history


# =============================================================================
# 10. Sensitivity analysis
# =============================================================================

def sensitivity_dataset_keys(tasks: OrderedDict) -> List[str]:
    preferred = [
        "breast_cancer_sklearn",
        "pima_diabetes",
        "ionosphere",
        "syn_moons",
        "syn_noisy_overlap",
    ]
    keys = [k for k in preferred if k in tasks]
    if len(keys) >= 5:
        return keys[:5]
    for k in tasks.keys():
        if k not in keys:
            keys.append(k)
        if len(keys) >= 5:
            break
    return keys


def sensitivity_candidates() -> List[Tuple[str, str, MethodConfig]]:
    out = []

    # CA-MSL tau sweep
    base_ca = next(m for m in BASE_METHODS if m.key == "ca_msl")
    for tau in [0.20, 0.35, 0.50, 0.75]:
        out.append(("ca_msl", "tau", clone_method(base_ca, _cid("sens_tau", tau=tau), tau=tau, beta=4.5, alpha=0.30)))

    # CA-MSL alpha sweep
    for alpha in [0.00, 0.15, 0.30, 0.50]:
        out.append(("ca_msl", "alpha", clone_method(base_ca, _cid("sens_alpha", alpha=alpha), tau=0.35, beta=4.5, alpha=alpha)))

    # CA-MSL beta sweep
    for beta in [2.5, 4.5, 6.5, 8.0]:
        out.append(("ca_msl", "beta", clone_method(base_ca, _cid("sens_beta", beta=beta), tau=0.35, beta=beta, alpha=0.30)))

    # InfoStab-CA-MSL lambda scale sweep
    base_info = next(m for m in BASE_METHODS if m.key == "infostab_ca_msl")
    for scale in [0.10, 0.25, 0.50, 1.00, 2.00]:
        out.append(("infostab_ca_msl", "lambda_scale", with_lambda_scale(base_info, scale, tau=0.35, beta=4.5, alpha=0.30)))

    # InfoStab-CA-MSL tau/alpha checks at lighter regularization.
    for tau in [0.25, 0.35, 0.50]:
        out.append(("infostab_ca_msl", "tau_lightreg", with_lambda_scale(base_info, 0.25, tau=tau, beta=4.5, alpha=0.30)))
    for alpha in [0.15, 0.30, 0.50]:
        out.append(("infostab_ca_msl", "alpha_lightreg", with_lambda_scale(base_info, 0.25, tau=0.35, beta=4.5, alpha=alpha)))


    # New class-balanced / boundary-aware loss sensitivity.
    base_cb = next((m for m in BASE_METHODS if m.key == "cb_focal_ca_msl_sg"), None)
    if base_cb is not None:
        for gamma in [0.0, 0.5, 1.0, 1.5, 2.0]:
            out.append(("cb_focal_ca_msl_sg", "hard_gamma", clone_method(
                base_cb,
                _cid("sens_hardgamma", gamma=gamma),
                tau=0.35, beta=4.5, alpha=0.30,
                class_balance_power=0.50, hard_gamma=gamma, minority_tau_multiplier=1.10
            )))
        for power in [0.0, 0.25, 0.50, 0.75, 1.0]:
            out.append(("cb_focal_ca_msl_sg", "class_balance_power", clone_method(
                base_cb,
                _cid("sens_cbpower", power=power),
                tau=0.35, beta=4.5, alpha=0.30,
                class_balance_power=power, hard_gamma=1.0, minority_tau_multiplier=1.10
            )))
        for mult in [1.0, 1.05, 1.10, 1.20]:
            out.append(("cb_focal_ca_msl_sg", "minority_tau_multiplier", clone_method(
                base_cb,
                _cid("sens_taumult", mult=mult),
                tau=0.35, beta=4.5, alpha=0.30,
                class_balance_power=0.50, hard_gamma=1.0, minority_tau_multiplier=mult
            )))

    base_boundary = next((m for m in BASE_METHODS if m.key == "boundary_cb_focal_ca_msl_sg"), None)
    if base_boundary is not None:
        for lam in [0.0, 0.10, 0.25, 0.40, 0.60]:
            out.append(("boundary_cb_focal_ca_msl_sg", "boundary_lambda", clone_method(
                base_boundary,
                _cid("sens_boundary", lam=lam),
                tau=0.35, beta=4.5, alpha=0.30,
                class_balance_power=0.50, hard_gamma=1.0, minority_tau_multiplier=1.10,
                boundary_lambda=lam, boundary_rho_power=1.0, boundary_entropy_power=1.0
            )))


    return out


def run_sensitivity_analysis_fn(tasks: OrderedDict) -> pd.DataFrame:
    if not CFG.run_sensitivity_analysis:
        return pd.DataFrame()

    keys = sensitivity_dataset_keys(tasks)
    cands = sensitivity_candidates()
    rows = []

    print("\n" + "=" * 100)
    print("RUNNING SENSITIVITY ANALYSIS")
    print("Datasets:", keys)
    print("Candidates:", len(cands))
    print("=" * 100)

    for key in keys:
        task = tasks[key]
        split = prepare_split(task, CFG.sensitivity_run)
        init_state = create_initial_state(split)

        print(f"\nSensitivity dataset={key} source={task['source']}")

        for method_family, factor, cand in cands:
            try:
                t0 = time.time()
                res, _ = train_candidate(
                    split,
                    init_state,
                    cand,
                    epochs=CFG.sensitivity_epochs,
                    patience=CFG.sensitivity_patience,
                    stage="sensitivity",
                    collect_history=False
                )
                res["sensitivity_family"] = method_family
                res["sensitivity_factor"] = factor
                res["time_sec"] = time.time() - t0
                rows.append(res)

                print(
                    f"  {method_family:<18} {factor:<14} {cand.candidate_id:<30} "
                    f"valF1={res['val_f1']:.4f} testF1={res['test_f1']:.4f} "
                    f"AUC={res['test_auc']:.4f}"
                )
            except Exception as e:
                print(f"  [ERROR] {key} {method_family} {factor} {cand.candidate_id}: {e}")

    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "sensitivity_analysis.csv", index=False)
    return df


# =============================================================================
# 11. Aggregation, statistical tests, and figures
# =============================================================================

def aggregate_results(raw: pd.DataFrame):
    metric_cols = [
        "test_f1", "test_macro_f1", "test_auc", "test_mcc",
        "test_balanced_accuracy", "final_sigma", "active_centers",
        "dead_centers", "time_sec"
    ]

    agg = raw.groupby(["method", "display_name", "source"])[metric_cols].agg(["mean", "std", "count"]).reset_index()
    agg.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in agg.columns]
    agg.to_csv(TABLE_DIR / "aggregate_by_source.csv", index=False)

    overall = raw.groupby(["method", "display_name"])[metric_cols].agg(["mean", "std", "count"]).reset_index()
    overall.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in overall.columns]
    overall = overall.sort_values("test_f1_mean", ascending=False)
    overall.to_csv(TABLE_DIR / "aggregate_overall.csv", index=False)

    selected_params = raw.groupby(["method", "candidate_id", "selected_params"]).size().reset_index(name="selected_count")
    selected_params = selected_params.sort_values(["method", "selected_count"], ascending=[True, False])
    selected_params.to_csv(TABLE_DIR / "selected_hyperparameters_frequency.csv", index=False)

    return agg, overall, selected_params


def holm_adjust(pvals: List[float]) -> List[float]:
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(n, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        adj = (n - rank) * pvals[idx]
        running = max(running, adj)
        adjusted[idx] = min(1.0, running)
    return adjusted.tolist()


def paired_tests(raw: pd.DataFrame, target: str = "ca_msl") -> pd.DataFrame:
    rows = []
    keys = ["dataset", "run"]
    target_df = raw[raw["method"] == target][keys + ["test_f1", "test_auc", "test_macro_f1", "test_mcc"]].rename(
        columns={
            "test_f1": "target_f1",
            "test_auc": "target_auc",
            "test_macro_f1": "target_macro_f1",
            "test_mcc": "target_mcc",
        }
    )

    for method in sorted(raw["method"].unique()):
        if method == target:
            continue
        other = raw[raw["method"] == method][keys + ["test_f1", "test_auc", "test_macro_f1", "test_mcc"]].rename(
            columns={
                "test_f1": "other_f1",
                "test_auc": "other_auc",
                "test_macro_f1": "other_macro_f1",
                "test_mcc": "other_mcc",
            }
        )
        merged = target_df.merge(other, on=keys, how="inner")
        if len(merged) < 5:
            continue
        for metric in ["f1", "auc", "macro_f1", "mcc"]:
            diff = merged[f"target_{metric}"] - merged[f"other_{metric}"]
            try:
                stat, p = wilcoxon(diff)
            except Exception:
                stat, p = np.nan, np.nan
            rows.append({
                "target": target,
                "baseline": method,
                "metric": metric,
                "n_pairs": len(diff),
                "mean_diff": float(np.mean(diff)),
                "median_diff": float(np.median(diff)),
                "wilcoxon_stat": stat,
                "p_value": p,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df["holm_p_value"] = np.nan
        for metric, idx in df.groupby("metric").groups.items():
            pvals = df.loc[idx, "p_value"].fillna(1.0).tolist()
            df.loc[idx, "holm_p_value"] = holm_adjust(pvals)
    df.to_csv(TABLE_DIR / f"paired_tests_vs_{target}.csv", index=False)
    return df


def make_latex_table(overall: pd.DataFrame):
    cols = [
        "method", "display_name",
        "test_f1_mean", "test_f1_std",
        "test_macro_f1_mean", "test_auc_mean",
        "test_mcc_mean", "active_centers_mean",
        "time_sec_mean"
    ]
    tab = overall[cols].copy()
    tex = tab.to_latex(index=False, float_format="%.4f")
    (TABLE_DIR / "main_results_table_validation_tuned.tex").write_text(tex, encoding="utf-8")
    return tex


def make_ablation_table(raw: pd.DataFrame):
    chain = ["msl", "ca_msl", "ca_msl_sg", "cb_focal_ca_msl_sg", "boundary_cb_focal_ca_msl_sg", "infostab_ca_msl"]
    sub = raw[raw["method"].isin(chain)].copy()
    metrics = ["test_f1", "test_macro_f1", "test_auc", "test_mcc", "final_sigma", "active_centers", "dead_centers"]
    table = sub.groupby("method")[metrics].agg(["mean", "std", "count"]).reset_index()
    table.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in table.columns]
    table.to_csv(TABLE_DIR / "ablation_table_msl_to_infostab.csv", index=False)
    return table


def plot_ranking(overall: pd.DataFrame):
    if not CFG.make_figures or overall.empty:
        return

    top = overall.sort_values("test_f1_mean", ascending=True)
    plt.figure(figsize=(10, 7))
    plt.barh(top["method"], top["test_f1_mean"])
    plt.xlabel("Mean test F1")
    plt.title("Validation-tuned method ranking, no GATE")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ranking_f1_validation_tuned_no_gate.png", dpi=220)
    plt.close()

    top_auc = overall.sort_values("test_auc_mean", ascending=True)
    plt.figure(figsize=(10, 7))
    plt.barh(top_auc["method"], top_auc["test_auc_mean"])
    plt.xlabel("Mean test AUC")
    plt.title("Validation-tuned method ranking by AUC, no GATE")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ranking_auc_validation_tuned_no_gate.png", dpi=220)
    plt.close()


def plot_source_comparison(raw: pd.DataFrame):
    if not CFG.make_figures or raw.empty:
        return
    for source in sorted(raw["source"].unique()):
        sub = raw[raw["source"] == source]
        rank = sub.groupby("method")["test_f1"].mean().sort_values()
        plt.figure(figsize=(10, 7))
        plt.barh(rank.index, rank.values)
        plt.xlabel("Mean test F1")
        plt.title(f"Validation-tuned ranking on {source} datasets")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"ranking_f1_{source}_validation_tuned_no_gate.png", dpi=220)
        plt.close()


def plot_selected_hyperparameters(selected_params: pd.DataFrame):
    if not CFG.make_figures or selected_params.empty:
        return

    for method in selected_params["method"].unique():
        sub = selected_params[selected_params["method"] == method].head(12)
        if sub.empty:
            continue
        labels = sub["candidate_id"].astype(str).values
        vals = sub["selected_count"].values
        plt.figure(figsize=(10, 4))
        plt.bar(range(len(vals)), vals)
        plt.xticks(range(len(vals)), labels, rotation=45, ha="right")
        plt.ylabel("Selection count")
        plt.title(f"Selected validation-tuned hyperparameters: {method}")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"selected_hyperparams_{method}.png", dpi=220)
        plt.close()


def plot_sensitivity(sens: pd.DataFrame):
    if not CFG.make_figures or sens.empty:
        return

    for (family, factor), sub in sens.groupby(["sensitivity_family", "sensitivity_factor"]):
        pivot = sub.groupby("candidate_id")["test_f1"].mean().sort_index()
        plt.figure(figsize=(9, 4))
        plt.plot(range(len(pivot)), pivot.values, marker="o")
        plt.xticks(range(len(pivot)), pivot.index, rotation=45, ha="right")
        plt.ylabel("Mean test F1")
        plt.title(f"Sensitivity: {family} / {factor}")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"sensitivity_{family}_{factor}.png", dpi=220)
        plt.close()


def package_outputs():
    zip_path = Path(f"/content/{CFG.output_name}_{TIMESTAMP}.zip") if Path("/content").exists() else Path(f"./{CFG.output_name}_{TIMESTAMP}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT_ROOT.rglob("*"):
            if p.is_file():
                z.write(p, arcname=str(p.relative_to(OUTPUT_ROOT)))
    print("ZIP:", zip_path)
    return zip_path


# =============================================================================
# 12. One-click main
# =============================================================================

def run_sanity_check():
    print("\n" + "=" * 80)
    print("LOSS SANITY CHECK")
    print("=" * 80)
    scores = torch.tensor([-2.0, -0.5, 0.5, 2.0], dtype=torch.float32, device=DEVICE)
    y = torch.tensor([0.0, 0.0, 1.0, 1.0], dtype=torch.float32, device=DEVICE)
    phi = torch.rand((len(scores), 6), dtype=torch.float32, device=DEVICE).clamp_min(1e-4)
    aux = {"phi": phi, "sigma": torch.tensor(1.0, device=DEVICE), "sigma0": torch.tensor(1.0, device=DEVICE)}
    for m in BASE_METHODS:
        try:
            if m.base_loss in ["cb_focal_ca_msl_sg", "boundary_cb_focal_ca_msl_sg"]:
                val, _ = class_balanced_hard_ca_msl_sg_loss(scores, y, aux, m)
            else:
                val = scalar_loss(scores, y, m)
            n_cands = len(get_validation_candidates(m))
            print(f"{m.key:<36} loss={float(val.detach().cpu()):.6f} candidates={n_cands}")
        except Exception as e:
            print(f"{m.key:<36} ERROR {e}")


def main():
    print("\n" + "=" * 100)
    print("InfoStab / Boundary-Class-Balanced CA-MSL-SG Q1 Validation-Tuned Benchmark, NO-GATE")
    print("=" * 100)
    print("Device:", DEVICE)
    print("Runs:", CFG.n_runs)
    print("Tuning mode:", CFG.tuning_mode)
    print("Tuning epochs:", CFG.tuning_epochs, "| patience:", CFG.tuning_patience)
    print("Methods without GATE, including two new class-balanced / boundary-aware losses:", len(BASE_METHODS))
    print("Sensitivity:", CFG.run_sensitivity_analysis)
    print("Dataset profile: medical / biomedical first, then complex/noisy binary tasks")
    print("Output:", OUTPUT_ROOT)
    print("=" * 100)

    config_path = OUTPUT_ROOT / "config.json"
    config_path.write_text(json.dumps(asdict(CFG), indent=2, ensure_ascii=False), encoding="utf-8")

    run_sanity_check()

    tasks, manifest = load_all_datasets()
    print("\nLoaded tasks:", len(tasks))
    print(manifest)

    raw, candidate_df, history = run_full_experiment(tasks)
    print("\nRaw selected result shape:", raw.shape)
    print("Candidate result shape:", candidate_df.shape)

    agg_by_source, overall, selected_params = aggregate_results(raw)

    print("\nOVERALL VALIDATION-TUNED RANKING")
    cols = ["method", "display_name", "test_f1_mean", "test_f1_std", "test_auc_mean", "test_macro_f1_mean", "test_mcc_mean"]
    print(overall[cols])

    tex = make_latex_table(overall)
    ablation = make_ablation_table(raw)

    tests_ca = paired_tests(raw, target="ca_msl")
    tests_sg = paired_tests(raw, target="ca_msl_sg")
    tests_cb_focal = paired_tests(raw, target="cb_focal_ca_msl_sg")
    tests_boundary_cb = paired_tests(raw, target="boundary_cb_focal_ca_msl_sg")
    tests_msl = paired_tests(raw, target="msl")
    tests_info = paired_tests(raw, target="infostab_ca_msl")

    print("\nPaired tests vs CA-MSL:")
    if not tests_ca.empty:
        print(tests_ca.sort_values(["metric", "holm_p_value"]).head(20))

    sens = run_sensitivity_analysis_fn(tasks)

    plot_ranking(overall)
    plot_source_comparison(raw)
    plot_selected_hyperparameters(selected_params)
    plot_sensitivity(sens)

    zip_path = package_outputs() if CFG.auto_zip else None

    print("\nDONE.")
    print("Output directory:", OUTPUT_ROOT)
    print("ZIP package:", zip_path)
    return {
        "tasks": tasks,
        "manifest": manifest,
        "raw": raw,
        "candidate_results": candidate_df,
        "history": history,
        "aggregate": overall,
        "selected_params": selected_params,
        "ablation": ablation,
        "tests_ca": tests_ca,
        "tests_sg": tests_sg,
        "tests_cb_focal": tests_cb_focal,
        "tests_boundary_cb": tests_boundary_cb,
        "tests_msl": tests_msl,
        "tests_info": tests_info,
        "sensitivity": sens,
        "zip_path": str(zip_path) if zip_path else None,
    }


RUN_OUTPUTS = main()


Streaming output truncated to the last 5000 lines.
[1167/1875] Tuning method=ca_msl_sg, candidates=4
    cand 01/04 ca_msl_sg_alpha0p15_beta3_tau0p25  valF1=0.8860 testF1=0.8663 AUC=0.9476 σ=3.647 time=1.5s
    cand 02/04 ca_msl_sg_alpha0p15_beta4p5_tau0p35 valF1=0.8726 testF1=0.8715 AUC=0.9494 σ=3.648 time=1.6s
    cand 03/04 ca_msl_sg_alpha0p3_beta4p5_tau0p35 valF1=0.8724 testF1=0.8697 AUC=0.9498 σ=3.644 time=1.6s
    cand 04/04 ca_msl_sg_alpha0p4_beta4p5_tau0p5  valF1=0.8756 testF1=0.8734 AUC=0.9489 σ=3.669 time=1.6s
    SELECTED ca_msl_sg_alpha0p15_beta3_tau0p25 | valF1=0.8860 testF1=0.8663 macroF1=0.8875 AUC=0.9476

[1168/1875] Tuning method=cb_focal_ca_msl_sg, candidates=6
    cand 01/06 cb_focal_ca_msl_sg_alpha0p15_beta3_class_balance_power0p5_hard_gamma0p5_minority_tau_multiplier1_tau0p25 valF1=0.8766 testF1=0.8697 AUC=0.9538 σ=3.653 time=2.3s
    cand 02/06 cb_focal_ca_msl_sg_alpha0p15_beta3_class_balance_power0p5_hard_gamma1_minority_tau_multiplier1p05_tau0p25 valF1=0.8734 te